## Nunchaku w4x4y16 (NVFP4) 打包与形状查看

本笔记随机生成 Linear 层的张量（per-tensor scale、每行16分组的 subscale、bias、smooth、LoRA），
调用 `convert_to_nunchaku_w4x4y16_linear_weight(..., float_point=True)` 进行打包，并打印输入与打包后张量的形状/类型。


In [3]:
# 从指定路径直接读取已打包文件并查看内容
from pathlib import Path
import torch

pt_path = Path('/FirstIntelligence/home/zihanm/deepcompressor/nunchaku_linear_w4x4y16_nvfp4.pt')

state = torch.load(str(pt_path), map_location='cpu', weights_only=False)

print('Loaded from:', pt_path)
print('Keys:', list(state.keys()))

def shape_dtype(x):
    return (tuple(x.shape), x.dtype) if isinstance(x, torch.Tensor) else type(x)

for key in state.keys():
    val = state.get(key)
    print(f'{key}:', 'None' if val is None else shape_dtype(val))

print('meta:', state.get('meta', {}))


Loaded from: /FirstIntelligence/home/zihanm/deepcompressor/nunchaku_linear_w4x4y16_nvfp4.pt
Keys: ['weight', 'scale', 'bias', 'smooth', 'subscale', 'lora_down', 'lora_up', 'orig_weight', 'orig_scale', 'orig_bias', 'orig_smooth', 'orig_subscale_base_blocks', 'orig_subscale_forward', 'orig_lora_down', 'orig_lora_up', 'weight_bw', 'scale_bw', 'bias_bw', 'smooth_bw', 'subscale_bw', 'lora_up_bw', 'lora_down_bw', 'orig_weight_T', 'orig_subscale_bw', 'orig_smooth_bw', 'orig_lora_up_T', 'orig_lora_down_T', 'meta']
weight: ((256, 64), torch.int8)
scale: ((1,), torch.bfloat16)
bias: ((256,), torch.bfloat16)
smooth: ((128,), torch.bfloat16)
subscale: ((8, 256), torch.float8_e4m3fn)
lora_down: ((128, 16), torch.bfloat16)
lora_up: ((256, 16), torch.bfloat16)
orig_weight: ((256, 128), torch.bfloat16)
orig_scale: ((1,), torch.bfloat16)
orig_bias: ((256,), torch.bfloat16)
orig_smooth: ((128,), torch.bfloat16)
orig_subscale_base_blocks: ((16, 1, 8, 1), torch.bfloat16)
orig_subscale_forward: ((256, 1, 8

In [3]:
# 查看 safetensors 文件中的所有张量形状/类型
from pathlib import Path

try:
    from safetensors.torch import load_file
except ImportError as e:
    raise ImportError("请先安装 safetensors: pip install safetensors") from e

sf_path = Path('/data/zihan/dev_nvfp4_nunchaku/flux.1-dev/transformer_blocks.safetensors')
assert sf_path.exists(), f'文件不存在: {sf_path}'

state = load_file(str(sf_path))  # 加载到 CPU

print('Loaded:', sf_path)
print('Num tensors:', len(state))

total_elems = 0
total_bytes = 0
for k in sorted(state.keys()):
    t = state[k]
    total_elems += t.numel()
    total_bytes += t.element_size() * t.numel()
    print(f'{k}: shape={tuple(t.shape)}, dtype={t.dtype}')

print('Total elements:', total_elems)
print('Total bytes (approx):', total_bytes)


Loaded: /data/zihan/dev_nvfp4_nunchaku/flux.1-dev/transformer_blocks.safetensors
Num tensors: 2888
single_transformer_blocks.0.mlp_fc1.bias: shape=(12288,), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc1.lora_down: shape=(3072, 32), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc1.lora_up: shape=(12288, 32), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc1.qweight: shape=(12288, 1536), dtype=torch.int8
single_transformer_blocks.0.mlp_fc1.smooth: shape=(3072,), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc1.smooth_orig: shape=(3072,), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc1.wscales: shape=(192, 12288), dtype=torch.float8_e4m3fn
single_transformer_blocks.0.mlp_fc1.wtscale: shape=(1,), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc2.bias: shape=(3072,), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc2.lora_down: shape=(12288, 32), dtype=torch.bfloat16
single_transformer_blocks.0.mlp_fc2.lora_up: shape=(3072, 32), dtype=